# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [2]:
!pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [omnivoice]


## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [2]:
!omnivoice-demo --share

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
2026-11-10 14:29:52,582 root INFO: Loading model from k2-fsa/OmniVoice, device=cpu ...
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2

## 3. Option B — Python API

### 3.1 Load Model

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [4]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

Upload a reference audio file (wav/mp3/flac):


Saving VozV.wav to VozV.wav
Uploaded: VozV.wav


In [6]:
audio = model.generate(
    text="no tengo el chichi para farolillos",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

NameError: name 'model' is not defined

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

In [3]:

import torch
import inspect
import omnivoice
from omnivoice import OmniVoice

print("=== OMNIVOICE ===")
print("Módulo:", omnivoice.__file__)
print("Carga del modelo:", inspect.signature(OmniVoice.from_pretrained))
print("Generación:", inspect.signature(OmniVoice.generate))

print("\n=== GPU ===")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA no disponible")

print("\n=== MEMORIA GPU ===")
if torch.cuda.is_available():
    print("Total:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
    print("Libre:", round(torch.cuda.mem_get_info()[0] / 1024**3, 2), "GB")


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


=== OMNIVOICE ===
Módulo: /usr/local/lib/python3.13/dist-packages/omnivoice/__init__.py
Carga del modelo: (pretrained_model_name_or_path, *args, **kwargs)
Generación: (self, text: Union[str, list[str]], language: Union[str, list[str], NoneType] = None, ref_text: Union[str, list[str], NoneType] = None, ref_audio: Union[str, list[str], tuple[torch.Tensor, int], list[tuple[torch.Tensor, int]], NoneType] = None, voice_clone_prompt: Union[omnivoice.models.omnivoice.VoiceClonePrompt, list[omnivoice.models.omnivoice.VoiceClonePrompt], NoneType] = None, instruct: Union[str, list[str], NoneType] = None, duration: Union[float, list[Optional[float]], NoneType] = None, speed: Union[float, list[Optional[float]], NoneType] = None, generation_config: Optional[omnivoice.models.omnivoice.OmniVoiceGenerationConfig] = None, normalize_text: bool = False, **kwargs) -> list[numpy.ndarray]

=== GPU ===
Tesla T4

=== MEMORIA GPU ===
Total: 14.56 GB
Libre: 14.46 GB


In [4]:

import omnivoice
from pathlib import Path

root = Path(omnivoice.__file__).parent
print("=== UBICACIÓN ===")
print(root)

print("\n=== CÓDIGO DE CARGA Y GENERACIÓN ===")
terms = ("OmniVoice.from_pretrained", "model.generate(", "ref_audio", "ref_text")

for file in root.rglob("*.py"):
    try:
        lines = file.read_text(encoding="utf-8").splitlines()
    except Exception:
        continue

    matches = [i for i, line in enumerate(lines) if any(t in line for t in terms)]
    if matches:
        print(f"\n--- {file.name} ---")
        shown = set()
        for i in matches:
            for j in range(max(0, i-5), min(len(lines), i+9)):
                if j not in shown:
                    print(f"{j+1}: {lines[j]}")
                    shown.add(j)

print("\n=== CONFIGURACIÓN DE GENERACIÓN ===")
from omnivoice.models.omnivoice import OmniVoiceGenerationConfig
import inspect
print(inspect.signature(OmniVoiceGenerationConfig))


=== UBICACIÓN ===
/usr/local/lib/python3.13/dist-packages/omnivoice

=== CÓDIGO DE CARGA Y GENERACIÓN ===

--- duration.py ---
206:         return sum(self._get_char_weight(c) for c in text)
207: 
208:     def estimate_duration(
209:         self,
210:         target_text: str,
211:         ref_text: str,
212:         ref_duration: float,
213:         low_threshold: Optional[float] = 50,
214:         boost_strength: float = 3,
215:     ) -> float:
216:         """
217: 
218:         Args:
219:             target_text (str): The text for which we want to estimate the duration.
220:             ref_text (str): The reference text that was used to measure
221:                 the ref_duration.
222:             ref_duration (float): The actual duration it took
223:                 to speak the ref_text.
224:             low_threshold (float): The minimum duration threshold below which the
225:                 estimation will be considered unreliable.
226:             boost_strength (float):

In [5]:

import omnivoice
from pathlib import Path

root = Path(omnivoice.__file__).parent

for file in root.rglob("*.py"):
    try:
        lines = file.read_text(encoding="utf-8").splitlines()
    except Exception:
        continue

    for i, line in enumerate(lines):
        if "def generate(" in line or "def from_pretrained(" in line:
            print(f"\n=== {file.name}, línea {i+1} ===")
            for j in range(i, min(i+35, len(lines))):
                print(f"{j+1}: {lines[j]}")



=== omnivoice.py, línea 333 ===
333:     def from_pretrained(cls, pretrained_model_name_or_path, *args, **kwargs):
334:         train_mode = kwargs.pop("train", False)
335:         load_asr = kwargs.pop("load_asr", False)
336:         asr_model_name = kwargs.pop("asr_model_name", None)
337:         asr_device = kwargs.pop("asr_device", None)
338: 
339:         # Suppress noisy INFO logs from transformers/huggingface_hub during loading
340:         _prev_disable = logging.root.manager.disable
341:         logging.disable(logging.INFO)
342: 
343:         try:
344:             # Resolve to local path first; download only if not already cached
345:             resolved_path = _resolve_model_path(pretrained_model_name_or_path)
346: 
347:             model = super().from_pretrained(resolved_path, *args, **kwargs)
348: 
349:             if not train_mode:
350:                 model.text_tokenizer = AutoTokenizer.from_pretrained(resolved_path)
351: 
352:                 audio_tokenizer_path =

In [6]:

import inspect
from omnivoice import OmniVoice
from omnivoice.models.omnivoice import OmniVoiceGenerationConfig

print("=== PREPARACIÓN DE VOZ ===")
if hasattr(OmniVoice, "create_voice_clone_prompt"):
    print(inspect.signature(OmniVoice.create_voice_clone_prompt))
    print(inspect.getdoc(OmniVoice.create_voice_clone_prompt))

print("\n=== CONFIGURACIÓN ===")
print(inspect.signature(OmniVoiceGenerationConfig))

print("\n=== FINAL DE LA CARGA DEL MODELO ===")
source = inspect.getsource(OmniVoice.from_pretrained)
print("\n".join(source.splitlines()[-30:]))


=== PREPARACIÓN DE VOZ ===
(self, ref_audio: Union[str, tuple[torch.Tensor, int]], ref_text: Optional[str] = None, preprocess_prompt: bool = True) -> omnivoice.models.omnivoice.VoiceClonePrompt
Create a reusable voice clone prompt from reference audio.

Args:
    ref_audio: File path (str) or ``(waveform, sample_rate)`` tuple.
        waveform should be a 1-D or 2-D torch.Tensor (channels x samples).
    ref_text: Transcript of the reference audio. If ``None``, the
        ASR model will be used to auto-transcribe (must call
        :meth:`load_asr_model` first).
    preprocess_prompt: If ``True`` (default), apply silence removal and
        trimming to the reference audio, add punctuation in the end
        of reference text (if not already)

Returns:
    A :class:`VoiceClonePrompt` that can be passed to :meth:`generate`.

=== CONFIGURACIÓN ===
(num_step: int = 32, guidance_scale: float = 2.0, t_shift: float = 0.1, layer_penalty_factor: float = 5.0, position_temperature: float = 5.0, 

In [7]:

%%writefile /content/servidor_omnivoice.py
import io
import os
import re
import time
import uuid
import hashlib
import threading
from collections import OrderedDict

import numpy as np
import soundfile as sf
import torch

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import Response, JSONResponse
from omnivoice import OmniVoice
from omnivoice.models.omnivoice import OmniVoiceGenerationConfig
import uvicorn


# --------------------------------------------------
# CONFIGURACIÓN
# --------------------------------------------------

MODEL_NAME = "k2-fsa/OmniVoice"
HOST = "0.0.0.0"
PORT = 8000
MAX_PROFILES = 8
MAX_CLIP_BYTES = 30 * 1024 * 1024
SERVICE = "ai-npc-omnivoice-server"
PROTOCOL = 1

INSTANCE = str(uuid.uuid4())
VERSION = "1.0.0"

app = FastAPI()
model = None
model_ready = False
model_error = None

# Referencias WAV recibidas del mod
clips = {}

# Tabla de perfiles recibida del mod
voice_table = {}

# Prompts de voz ya procesados
prompts = OrderedDict()

# Sincronización del acceso al modelo/GPU
model_lock = threading.Lock()


# --------------------------------------------------
# CARGA DEL MODELO
# --------------------------------------------------

def load_model():
    global model, model_ready, model_error

    try:
        print("Cargando OmniVoice...")
        model = OmniVoice.from_pretrained(
            MODEL_NAME,
            device_map="cuda:0" if torch.cuda.is_available() else "cpu",
            dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        model_ready = True
        print("OmniVoice listo.")

    except Exception as e:
        model_error = str(e)
        print("ERROR cargando OmniVoice:", repr(e))


# --------------------------------------------------
# VALIDACIONES
# --------------------------------------------------

def valid_key(key):
    return isinstance(key, str) and bool(
        re.fullmatch(r"[a-z0-9][a-z0-9-]{0,79}", key)
    )


def get_descriptor(profile):
    descriptor = voice_table.get(profile)

    if not isinstance(descriptor, dict):
        raise HTTPException(404, detail={
            "code": "unknown-profile",
            "message": "Perfil desconocido"
        })

    if "reference" not in descriptor:
        raise HTTPException(404, detail={
            "code": "unknown-profile",
            "message": "El perfil no tiene una referencia de voz"
        })

    return descriptor


def get_clip(profile):
    descriptor = get_descriptor(profile)
    expected_hash = descriptor.get("sha256")

    clip = clips.get(profile)

    if not clip or clip["sha256"] != expected_hash:
        raise HTTPException(404, detail={
            "code": "unknown-profile",
            "message": "Falta el audio de referencia"
        })

    return clip["bytes"], descriptor


# --------------------------------------------------
# CREACIÓN Y CACHÉ DE PROMPTS
# --------------------------------------------------

def build_prompt(profile):
    audio_bytes, descriptor = get_clip(profile)

    digest = descriptor["sha256"]

    cached = prompts.get(profile)

    if cached and cached["sha256"] == digest:
        prompts.move_to_end(profile)
        return cached["prompt"]

    try:
        audio, sample_rate = sf.read(
            io.BytesIO(audio_bytes),
            dtype="float32",
            always_2d=False
        )

        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        waveform = torch.from_numpy(
            np.asarray(audio, dtype=np.float32).copy()
        )

        transcript = descriptor.get("transcript", "")

        prompt = model.create_voice_clone_prompt(
            ref_audio=(waveform, sample_rate),
            ref_text=transcript,
            preprocess_prompt=True
        )

        prompts[profile] = {
            "sha256": digest,
            "prompt": prompt
        }

        prompts.move_to_end(profile)

        while len(prompts) > MAX_PROFILES:
            prompts.popitem(last=False)

        return prompt

    except HTTPException:
        raise
    except Exception as e:
        print("Error preparando perfil:", profile, repr(e))
        raise HTTPException(500, detail={
            "code": "voice-prepare-failed",
            "message": str(e)
        })


# --------------------------------------------------
# RESPUESTAS DE ERROR
# --------------------------------------------------

def error_response(code, message, status):
    return JSONResponse(
        status_code=status,
        content={
            "error": {
                "code": code,
                "message": message
            }
        }
    )


# --------------------------------------------------
# HEALTH
# --------------------------------------------------

@app.get("/v1/health")
def health():
    if model_error:
        return {
            "service": SERVICE,
            "protocol": PROTOCOL,
            "state": "error",
            "detail": model_error[:120],
            "instance": INSTANCE,
            "version": VERSION
        }

    return {
        "service": SERVICE,
        "protocol": PROTOCOL,
        "state": "ready" if model_ready else "loading",
        "instance": INSTANCE,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "version": VERSION
    }


# --------------------------------------------------
# SINCRONIZACIÓN DE PERFILES
# --------------------------------------------------

@app.post("/v1/voices")
async def receive_voices(request: Request):
    global voice_table

    data = await request.json()
    incoming = data.get("voices", {})

    if not isinstance(incoming, dict):
        return error_response(
            "invalid-voices",
            "La tabla de voces no es válida",
            400
        )

    missing = []

    for key, descriptor in incoming.items():
        if not valid_key(key):
            continue

        if not isinstance(descriptor, dict):
            continue

        if "reference" not in descriptor:
            continue

        reference = descriptor.get("reference", "")
        sha = descriptor.get("sha256", "")
        transcript = descriptor.get("transcript", "")

        if not isinstance(reference, str) or not reference.startswith(
            ("runtime/voice-references/", "runtime/player-voices/")
        ):
            continue

        if not re.fullmatch(r"[a-f0-9]{64}", sha):
            continue

        if not isinstance(transcript, str) or not transcript.strip():
            continue

        if len(transcript) > 12000:
            continue

        voice_table[key] = {
            "reference": reference,
            "sha256": sha,
            "transcript": transcript
        }

        clip = clips.get(key)

        if not clip or clip["sha256"] != sha:
            missing.append({"key": key})

    # Elimina perfiles que ya no están en la tabla recibida.
    active_keys = set(incoming.keys())

    for key in list(voice_table):
        if key not in active_keys:
            voice_table.pop(key, None)
            prompts.pop(key, None)
            clips.pop(key, None)

    return {
        "missing": missing,
        "prepared": list(prompts.keys())
    }


# --------------------------------------------------
# SUBIDA DE REFERENCIAS WAV
# --------------------------------------------------

@app.put("/v1/clips/{key}")
async def upload_clip(key: str, request: Request):
    if not valid_key(key):
        raise HTTPException(400, "Clave de perfil inválida")

    if key not in voice_table:
        raise HTTPException(404, "Perfil desconocido")

    descriptor = voice_table[key]
    expected = request.query_params.get("sha256", "")

    if expected != descriptor["sha256"]:
        raise HTTPException(400, "SHA256 no coincide")

    audio_bytes = await request.body()

    if not audio_bytes or len(audio_bytes) > MAX_CLIP_BYTES:
        raise HTTPException(413, "Audio vacío o demasiado grande")

    actual = hashlib.sha256(audio_bytes).hexdigest()

    if actual != expected:
        raise HTTPException(400, "SHA256 incorrecto")

    # Comprobar que realmente es WAV
    try:
        sf.info(io.BytesIO(audio_bytes))
    except Exception:
        raise HTTPException(400, "El archivo no es un WAV válido")

    clips[key] = {
        "sha256": actual,
        "bytes": audio_bytes
    }

    prompts.pop(key, None)

    return {"ok": True, "key": key}


# --------------------------------------------------
# PREPARACIÓN EXPLÍCITA
# --------------------------------------------------

@app.post("/v1/prepare")
async def prepare_voice(request: Request):
    if not model_ready:
        return error_response(
            "server-loading",
            "OmniVoice todavía está cargando",
            503
        )

    data = await request.json()
    profile = data.get("profile")
    steps = data.get("steps", 32)

    if not valid_key(profile):
        return error_response("invalid-profile", "Perfil inválido", 400)

    if steps not in (16, 32):
        return error_response("invalid-steps", "Calidad inválida", 400)

    started = time.perf_counter()

    try:
        with model_lock:
            build_prompt(profile)

        elapsed = (time.perf_counter() - started) * 1000

        return {
            "prepared": profile,
            "ms": round(elapsed, 2)
        }

    except HTTPException as e:
        return JSONResponse(status_code=e.status_code, content=e.detail)


# --------------------------------------------------
# SÍNTESIS
# --------------------------------------------------

@app.post("/v1/synthesize")
async def synthesize(request: Request):
    if not model_ready:
        return error_response(
            "server-loading",
            "OmniVoice todavía está cargando",
            503
        )

    data = await request.json()

    profile = data.get("profile")
    text = data.get("text", "")
    steps = data.get("steps", 32)
    language = data.get("language")

    if not valid_key(profile):
        return error_response("invalid-profile", "Perfil inválido", 400)

    if not isinstance(text, str) or not text.strip() or len(text) > 12000:
        return error_response("invalid-text", "Texto inválido", 400)

    if steps not in (16, 32):
        return error_response("invalid-steps", "Calidad inválida", 400)

    if language is not None:
        if not isinstance(language, str) or not re.fullmatch(
            r"[a-z]{2,3}", language
        ):
            return error_response(
                "invalid-language",
                "Idioma inválido",
                400
            )

    started = time.perf_counter()
    generation_ms = 0

    try:
        with model_lock:
            # El primer render prepara el prompt dentro de la síntesis.
            prompt = build_prompt(profile)

            generation_started = time.perf_counter()

            config = OmniVoiceGenerationConfig(
                num_step=steps
            )

            kwargs = {
                "text": text,
                "voice_clone_prompt": prompt,
                "generation_config": config
            }

            if language:
                kwargs["language"] = language

            result = model.generate(**kwargs)

            generation_ms = (
                time.perf_counter() - generation_started
            ) * 1000

            if not result:
                raise RuntimeError("OmniVoice no ha generado audio")

            audio = np.asarray(result[0], dtype=np.float32)

            if audio.ndim > 1:
                audio = np.squeeze(audio)

            wav_buffer = io.BytesIO()

            sf.write(
                wav_buffer,
                audio,
                model.sampling_rate,
                format="WAV",
                subtype="PCM_16"
            )

            wav_bytes = wav_buffer.getvalue()

        elapsed = (time.perf_counter() - started) * 1000

        timing = {
            "prepared": profile,
            "voiceGenerationMs": round(generation_ms, 2),
            "voiceRuntime": "omnivoice",
            "voiceAcceleration": "eager",
            "voiceDtype": "float16" if torch.cuda.is_available() else "float32"
        }

        headers = {
            "Content-Type": "audio/wav",
            "X-OmniVoice-Timing": __import__("json").dumps(timing)
        }

        print(
            f"[SYNTH] {profile} | {len(text)} caracteres | "
            f"{generation_ms:.0f} ms generación | "
            f"{elapsed:.0f} ms total"
        )

        return Response(
            content=wav_bytes,
            media_type="audio/wav",
            headers=headers
        )

    except HTTPException as e:
        return JSONResponse(status_code=e.status_code, content=e.detail)

    except Exception as e:
        print("ERROR SYNTHESIS:", repr(e))
        return error_response(
            "synthesis-failed",
            str(e)[:500],
            500
        )


# --------------------------------------------------
# ARRANQUE
# --------------------------------------------------

if __name__ == "__main__":
    load_model()

    uvicorn.run(
        app,
        host=HOST,
        port=PORT,
        log_level="info"
    )


Writing /content/servidor_omnivoice.py


In [8]:
import subprocess
import time
import requests

server = subprocess.Popen(
    ["python", "-u", "/content/servidor_omnivoice.py"],
    stdout=open("/content/omnivoice-server.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Servidor iniciado. PID:", server.pid)

for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/v1/health", timeout=3)
        data = r.json()

        print(data)

        if data.get("state") == "ready":
            print("OMNIVOICE LISTO")
            break

        if data.get("state") == "error":
            print("ERROR:", data.get("detail"))
            break

    except Exception:
        pass

    time.sleep(2)
else:
    print("El servidor no ha terminado de cargar.")

Servidor iniciado. PID: 8522
{'service': 'ai-npc-omnivoice-server', 'protocol': 1, 'state': 'ready', 'instance': 'f4889d4c-3233-4e4c-8d07-e6ef798b4ba4', 'device': 'cuda', 'version': '1.0.0'}
OMNIVOICE LISTO


In [9]:
import requests

r = requests.get("http://127.0.0.1:8000/v1/health")
print("HTTP:", r.status_code)
print(r.json())

HTTP: 200
{'service': 'ai-npc-omnivoice-server', 'protocol': 1, 'state': 'ready', 'instance': 'f4889d4c-3233-4e4c-8d07-e6ef798b4ba4', 'device': 'cuda', 'version': '1.0.0'}


In [10]:
import os
import requests

print("Servidor Python:")
try:
    print(requests.get("http://127.0.0.1:8000/v1/health", timeout=3).json())
except Exception as e:
    print("No está activo")

print("\nGradio:")
try:
    print(requests.get("http://127.0.0.1:7860", timeout=3).status_code)
except Exception:
    print("No está activo")

print("\nOpenPort:")
print(os.path.exists("/root/.openport/bin/openport"))

print("\nArchivos:")
for f in ["/content/servidor_omnivoice.py", "/content/omnivoice-server.log"]:
    print(f, os.path.exists(f))

Servidor Python:
{'service': 'ai-npc-omnivoice-server', 'protocol': 1, 'state': 'ready', 'instance': 'f4889d4c-3233-4e4c-8d07-e6ef798b4ba4', 'device': 'cuda', 'version': '1.0.0'}

Gradio:
No está activo

OpenPort:
False

Archivos:
/content/servidor_omnivoice.py True
/content/omnivoice-server.log True


In [11]:
import subprocess

subprocess.run(
    "curl -sSf https://openport.pw/install.sh | sh",
    shell=True,
    check=True
)

print("OpenPort instalado.")

OpenPort instalado.


In [12]:
import subprocess
import time

tunnel = subprocess.Popen(
    ["/root/.openport/bin/openport", "8000", "--tcp"],
    stdout=open("/content/openport-server.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(4)
print(open("/content/openport-server.log").read())

Allocated permanent port: 35052
OpenPort 2.0 (Ctrl+C to quit)

Status       online
Forwarding   openport.pw:35052 -> 127.0.0.1:8000 (TCP)
Account      usr_3f8f1b5a



In [13]:
import socket
import requests

ip = socket.gethostbyname("openport.pw")
port = 35052

print("IPv4:", ip)
print("Puerto:", port)

url = f"http://{ip}:{port}/v1/health"

try:
    r = requests.get(url, timeout=15)
    print("HTTP:", r.status_code)
    print("Respuesta:", r.json())
except Exception as e:
    print("ERROR:", repr(e))

IPv4: 193.26.115.30
Puerto: 35052
HTTP: 200
Respuesta: {'service': 'ai-npc-omnivoice-server', 'protocol': 1, 'state': 'ready', 'instance': 'f4889d4c-3233-4e4c-8d07-e6ef798b4ba4', 'device': 'cuda', 'version': '1.0.0'}
